In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import dash
from dash import dcc, html
import plotly.graph_objs as go

Load prepared data

In [2]:
df_sentiment_path = "./savedData/df_sentiment.csv"

if os.path.exists(df_sentiment_path):
    df_sentiment = pd.read_csv(df_sentiment_path)
    df_sentiment['Date'] = pd.to_datetime(df_sentiment['Date'])
    print("Already Saved")
else:
    print("Start the data_loader to write file into savedData")

Already Saved


In [3]:
df_sentiment.head()

,Date,text,Sentiment,sentiment_score,avg_sentiment,next_day_price
0,2019-05-27,È appena uscito un nuovo video! LES CRYPTOMONN...,Positive,1,1.0,8719.961914
1,2019-05-27,Cardano: Digitize Currencies; EOS https://t.co...,Positive,1,1.0,8719.961914
2,2019-05-27,Another Test tweet that wasn't caught in the s...,Positive,1,1.0,8719.961914
3,2019-05-27,Current Crypto Prices! \n\nBTC: $8721.99 USD\n...,Positive,1,1.0,8719.961914
4,2019-05-27,Spiv (Nosar Baz): BITCOIN Is An Asset &amp; NO...,Positive,1,1.0,8719.961914


In [ ]:
df_sentiment.dtypes

Finbert laden

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('yiyanghkust/finbert-pretrain')
model = AutoModel.from_pretrained('yiyanghkust/finbert-pretrain')

In [ ]:
model.eval()

In [ ]:
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state[:, 0, :].squeeze().numpy()  # [CLS]-Token

In [ ]:
df_sample = df_sentiment.head(20000)

In [ ]:
embeddings = np.stack([get_bert_embedding(t) for t in df_sample['text']])

In [ ]:
# Combine embeddings, sentiment score and avg sentiment per day
X = np.hstack([embeddings,
               df_sample[['sentiment_score', 'avg_sentiment']].values])
# For now just the next day price
y = df_sample['next_day_price'].values

In [ ]:
# Zusammenführen zum Speichern
Xy_stack = np.hstack([X, y.reshape(-1, 1)])
Xy_df = pd.DataFrame(Xy_stack)
Xy_df.to_csv('./savedData/sample.csv', index=False)

Checks if the embeddings already been done.
If not it has to calculate them again.
The calculation takes a very long time especially when it´s performed on the cpu

In [ ]:
XY_path = "./savedData/sample.csv"

if os.path.exists(XY_path):
    # Load data if exists
    data = pd.read_csv(XY_path)
    # Split load data into X and y
    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values
    print("Features und Ziel aus Datei geladen.")
else:
    # Compute Embeddings
    embeddings = np.stack([get_bert_embedding(t) for t in df_sentiment['text']])
    # Build X and y
    X = np.hstack([embeddings,
                   df_sentiment[['sentiment_score', 'avg_sentiment']].values])
    y = df_sentiment['next_day_price'].values
    
    # Merge and save
    Xy_stack = np.hstack([X, y.reshape(-1, 1)])
    Xy_df = pd.DataFrame(Xy_stack)
    Xy_df.to_csv(XY_path, index=False)
    print("Features und Ziel neu berechnet und gespeichert.")

Random Forest Klassifikator trainieren

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

model = RandomForestRegressor()
model.fit(X_train, y_train)
score = model.score(X_test, y_test)
print("R^2 auf Testdaten:", score)

In [ ]:
y_pred = model.predict(X_test)
# Dash App erstellen
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1(f'Random Forest Regression - R² Score: {score:.4f}'),
    
    dcc.Graph(
        id='actual-vs-predicted',
        figure={
            'data': [
                go.Scatter(
                    x=y_test,
                    y=y_pred,
                    mode='markers',
                    marker=dict(size=8, color='blue', opacity=0.6),
                    name='Predictions'
                ),
                go.Scatter(
                    x=[min(y_test), max(y_test)],
                    y=[min(y_test), max(y_test)],
                    mode='lines',
                    line=dict(color='red', dash='dash'),
                    name='Perfect Prediction'
                )
            ],
            'layout': go.Layout(
                title='Actual vs Predicted Values',
                xaxis={'title': 'Actual Values'},
                yaxis={'title': 'Predicted Values'},
                hovermode='closest'
            )
        }
    ),
    
    dcc.Graph(
        id='residuals',
        figure={
            'data': [
                go.Scatter(
                    x=y_pred,
                    y=y_test - y_pred,
                    mode='markers',
                    marker=dict(size=8, color='green', opacity=0.6)
                )
            ],
            'layout': go.Layout(
                title='Residual Plot',
                xaxis={'title': 'Predicted Values'},
                yaxis={'title': 'Residuals'},
                shapes=[dict(type='line', x0=min(y_pred), x1=max(y_pred), 
                            y0=0, y1=0, line=dict(color='red', dash='dash'))]
            )
        }
    )
])

if __name__ == '__main__':
    app.run_server(debug=True, port=8050)